In [ ]:
import numpy as np
import os
import plotly.graph_objects as go
from matplotlib import pyplot as plt
from lib.peak import fwhm_to_sigma
from hardware.orbitrap.event import KOEvent
from scipy.signal import find_peaks, peak_widths

In [ ]:
data_path = "C:\\Orbi\\"

files = os.listdir(data_path)
filepaths = [os.path.join(data_path, file) for file in files]

In [ ]:
mzs = []
specs = []
peaks = []

for file in filepaths:
    try:
        ke = KOEvent(file)

        t = ke.t

        mz, spec = ke.get_spec()
        mzs.append(mz)
        specs.append(spec)
        peak, _ = find_peaks(spec, threshold=10)
        fwhm = peak_widths(spec, peak, rel_height=0.5)[0]
        sig = np.array([fwhm_to_sigma(fw) for fw in fwhm])
        peaks.append(peak)
        print("peak count: %s" % len(peak))

    except Exception as e:
        print(f"{file} failed: %s" % e)

In [ ]:
file_i = 15
print(f"Selected file: {os.path.basename(filepaths[15])}")

In [ ]:
spec = specs[15]
mz = mzs[15]
peak = peaks[15]

In [ ]:
fig = go.Figure(
    [go.Scatter(x=mz, y=spec), go.Scatter(x=mz[peak], y=spec[peak], mode="markers")]
)

fig.show(renderer="browser")

In [ ]:
peak_traces = []
peak_norm_traces = []

p_x = np.linspace(-30, 30, 601)
p_ys = []
p_mzs = []
p_fwhms = []
p_heis = []

for i, p in enumerate(peak):
    p_height = spec[p]
    # Select a narrow region (peak center +/- dmz) of the spectrum around the peak
    p_mz_center = mz[p]
    dmz = 0.01
    p_mz0 = p_mz_center - dmz
    p_mz1 = p_mz_center + dmz
    # Find indices
    p_i0 = np.argmin(np.abs(mz - p_mz0))  # left border
    p_i1 = np.argmin(np.abs(mz - p_mz1))  # right border
    p_i = p - p_i0  # center
    # Peak region
    p_mz = mz[p_i0:p_i1]
    p_spec = spec[p_i0:p_i1]

    if np.max(p_spec) > p_height:
        # 'p' is not the biggest peak in range, dismiss
        continue

    # Normalize mz around 0 and spec to range [0, 1]
    p_mz_norm = p_mz - p_mz_center
    p_spec_norm = p_spec / p_height

    peak_traces.append(go.Scatter(x=p_mz_norm, y=p_spec_norm))

    # Refine peak center by fitting a parabola to 5 data points around the peak center
    fit_x = p_mz_norm[(p_i - 2) : (p_i + 3)]
    fit_y = p_spec_norm[(p_i - 2) : (p_i + 3)]
    try:
        z, [residual], _, _, _ = np.polyfit(fit_x, fit_y, 2, full=True)
        # if residual > 0.01:
        #     print("bad fit: %s" %i)
        #     raise Exception
    except Exception as e:
        # Fitting failed
        print(e)
        continue
    fit = np.poly1d(z)
    # Find parabola vertex = peak top location
    vertex_x = -z[1] / (2 * z[0])
    vertex_y = fit(vertex_x)
    # Use vertex to refine peak position and height
    p_mz_norm -= vertex_x
    p_spec_norm /= vertex_y

    # Calculate peak width
    p_fwhm = peak_widths(p_spec_norm, [p_i], rel_height=0.5)
    # Peak width in "samples"
    p_fwhm_i = p_fwhm[0][0]
    # Get approximate for the width of one "sample" in the spectrum
    amu_per_i = np.min(np.diff(p_mz))
    # Convert peak width in "samples" to width in "amu"
    p_fwhm_amu = p_fwhm_i * amu_per_i
    p_mzs.append(p_mz_center)
    p_fwhms.append(p_fwhm_amu)
    p_heis.append(p_height)
    # Scale peak to width sigma=1
    p_sigma = fwhm_to_sigma(p_fwhm_amu)
    p_mz_norm /= p_sigma

    # Interpolate the normalized (both width and height) peak into predefined domain "p_x"
    p_y = np.interp(p_x, p_mz_norm, p_spec_norm, left=0, right=0)
    p_ys.append(p_y)

    peak_norm_traces.append(go.Scatter(x=p_x, y=p_y, name=f"peak {i}"))

# fig0 = go.Figure(peak_traces)
# fig0.show()

# Calculate median peak shape
p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)
peak_norm_traces.append(
    go.Scatter(x=p_x, y=p_median, line={"color": "red", "width": 3}, name="median")
)

fig = go.FigureWidget(
    peak_norm_traces, {"title": f"Median peak shape of {len(peak_norm_traces)-1} peaks"}
)


def toggle_trace_visibility(trace, points, selector):
    if not points.point_inds:
        return
    trace.visible = "legendonly"
    selected_traces = [
        trace.y
        for trace in fig.data
        if ("peak" in trace.name and trace.visible != "legendonly")
    ]
    fig.update_layout({"title": f"Median peak shape of {len(selected_traces)} peaks"})
    fig.update_traces(
        {"y": np.median(np.array(selected_traces), axis=0)},
        selector=lambda trace: trace.name == "median",
    )


[trace.on_click(toggle_trace_visibility) for trace in fig.data]
fig

In [ ]:
# Calculate peak resolution from peak width
p_R = np.array(p_mzs) / np.array(p_fwhms)

plt.scatter(p_mzs, p_R)

# Fit parabola to peak resolutions to find resolution function
R_p = np.polyfit(p_mzs, p_R, 2)
R_fit = np.poly1d(R_p)

plt.plot(p_mzs, R_fit(p_mzs), "r")

In [ ]:
plt.plot([r - R_fit(p_mzs[i]) for i, r in enumerate(p_R)])

In [ ]:
from lib.file_func import load_file

f = load_file("KLTOF2_2023.07.21-11h30m40s")

In [ ]:
plt.plot(f.mz, f.signal)